In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from app.rider import Rider
from app.critical_power import CriticalPowerModel

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Create a Rider Profile

Core rider parameters:
- **weight**: Total mass in kg (rider + bike + gear)
- **ftp**: Functional Threshold Power in watts
- **critical_power**: Sustainable power indefinitely (usually ~95% of FTP)
- **w_prime**: Anaerobic work capacity in joules (typically 15,000-25,000)

In [ ]:
# Define rider parameters
RIDER_WEIGHT = 75  # kg (body weight)
BIKE_WEIGHT = 8    # kg
GEAR_WEIGHT = 2    # kg (bottles, tools, etc.)
TOTAL_WEIGHT = RIDER_WEIGHT + BIKE_WEIGHT + GEAR_WEIGHT

FTP = 280  # watts
CRITICAL_POWER = int(FTP * 0.95)  # ~266W
W_PRIME = 20000  # joules

print(f"Rider Profile:")
print(f"  Total weight: {TOTAL_WEIGHT} kg")
print(f"  FTP: {FTP} W")
print(f"  Critical Power: {CRITICAL_POWER} W")
print(f"  W': {W_PRIME:,} J")
print(f"  W'/kg: {W_PRIME/RIDER_WEIGHT:.0f} J/kg")

In [ ]:
# Create rider instance (if Rider class exists)
try:
    rider = Rider(
        weight=TOTAL_WEIGHT,
        ftp=FTP,
        critical_power=CRITICAL_POWER,
        w_prime=W_PRIME
    )
    print(f"Rider created: {rider}")
except Exception as e:
    print(f"Rider class note: {e}")
    print("Continuing with manual calculations...")

## 2. Critical Power Model

The 2-parameter critical power model:
$$P(t) = CP + \frac{W'}{t}$$

Where:
- $P(t)$ = power sustainable for time $t$
- $CP$ = critical power (watts)
- $W'$ = anaerobic work capacity (joules)
- $t$ = time (seconds)

In [ ]:
def power_for_duration(duration_sec, cp, w_prime):
    """Calculate max sustainable power for a given duration."""
    return cp + (w_prime / duration_sec)

def duration_at_power(power, cp, w_prime):
    """Calculate how long you can sustain a given power."""
    if power <= cp:
        return float('inf')  # Sustainable indefinitely
    return w_prime / (power - cp)

# Test some key durations
test_durations = [
    (60, "1 min"),
    (180, "3 min"),
    (300, "5 min"),
    (600, "10 min"),
    (1200, "20 min"),
    (3600, "60 min")
]

print("Power-Duration Curve:")
for dur_sec, label in test_durations:
    power = power_for_duration(dur_sec, CRITICAL_POWER, W_PRIME)
    print(f"  {label:>8}: {power:.0f} W ({power/RIDER_WEIGHT:.1f} W/kg)")

In [ ]:
# Plot power-duration curve
durations = np.linspace(30, 7200, 500)  # 30 sec to 2 hours
powers = [power_for_duration(d, CRITICAL_POWER, W_PRIME) for d in durations]

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(durations/60, powers, 'b-', linewidth=2)
ax.axhline(y=CRITICAL_POWER, color='red', linestyle='--', label=f'CP = {CRITICAL_POWER}W')
ax.axhline(y=FTP, color='orange', linestyle=':', label=f'FTP = {FTP}W')

# Mark key points
for dur_sec, label in test_durations[:4]:
    power = power_for_duration(dur_sec, CRITICAL_POWER, W_PRIME)
    ax.plot(dur_sec/60, power, 'ko', markersize=8)
    ax.annotate(f'{label}\n{power:.0f}W', (dur_sec/60, power), 
                textcoords='offset points', xytext=(10, 5), fontsize=9)

ax.set_xlabel('Duration (minutes)')
ax.set_ylabel('Power (watts)')
ax.set_title('Power-Duration Curve (2-Parameter CP Model)')
ax.set_xlim(0, 120)
ax.set_ylim(200, 600)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. W' Depletion and Recovery

When riding above CP, W' depletes. Below CP, it recovers.

**Depletion rate**: $\frac{dW'}{dt} = CP - P$ (negative when P > CP)

**Recovery**: Exponential recovery with time constant τ (~550s typical)

In [ ]:
def simulate_w_prime(powers, durations, cp, w_prime_max, tau=550):
    """
    Simulate W' depletion and recovery.
    
    Args:
        powers: List of power values
        durations: List of duration for each power (seconds)
        cp: Critical power
        w_prime_max: Starting W'
        tau: Recovery time constant
    
    Returns:
        List of W' values at each time point
    """
    w_prime_current = w_prime_max
    w_prime_history = [w_prime_current]
    times = [0]
    
    t = 0
    for power, duration in zip(powers, durations):
        # For each second in this interval
        for _ in range(int(duration)):
            t += 1
            if power > cp:
                # Depletion
                w_prime_current -= (power - cp)
            else:
                # Recovery (simplified exponential)
                recovery_rate = (w_prime_max - w_prime_current) / tau
                w_prime_current += recovery_rate
            
            w_prime_current = max(0, min(w_prime_max, w_prime_current))
            w_prime_history.append(w_prime_current)
            times.append(t)
    
    return times, w_prime_history

# Simulate a sample effort: hard start, then settle
effort_powers = [350, 300, 250, 320, 240, 280]
effort_durations = [60, 120, 180, 90, 300, 60]  # seconds each

times, w_prime_values = simulate_w_prime(
    effort_powers, effort_durations, CRITICAL_POWER, W_PRIME
)

print(f"W' simulation:")
print(f"  Started at: {W_PRIME:,} J")
print(f"  Minimum: {min(w_prime_values):,.0f} J ({min(w_prime_values)/W_PRIME*100:.0f}%)")
print(f"  Final: {w_prime_values[-1]:,.0f} J ({w_prime_values[-1]/W_PRIME*100:.0f}%)")

In [ ]:
# Plot W' depletion
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Power profile
t = 0
for power, duration in zip(effort_powers, effort_durations):
    color = 'red' if power > CRITICAL_POWER else 'green'
    ax1.fill_between([t, t+duration], 0, power, alpha=0.3, color=color)
    ax1.plot([t, t+duration], [power, power], color=color, linewidth=2)
    t += duration

ax1.axhline(y=CRITICAL_POWER, color='black', linestyle='--', label=f'CP = {CRITICAL_POWER}W')
ax1.set_ylabel('Power (W)')
ax1.set_title('Power Profile (Red = above CP, depleting W\')')
ax1.legend()
ax1.set_ylim(0, 400)

# W' balance
ax2.fill_between(times, 0, w_prime_values, alpha=0.3, color='blue')
ax2.plot(times, w_prime_values, 'b-', linewidth=2)
ax2.axhline(y=W_PRIME, color='green', linestyle='--', alpha=0.5, label=f'W\' max = {W_PRIME:,}J')
ax2.axhline(y=0, color='red', linestyle='-', linewidth=2)

ax2.set_xlabel('Time (seconds)')
ax2.set_ylabel('W\' Balance (J)')
ax2.set_title('W\' Depletion and Recovery')
ax2.legend()
ax2.set_ylim(-1000, W_PRIME * 1.1)

plt.tight_layout()
plt.show()

## 4. Aerodynamic Drag (CdA)

CdA = Drag coefficient × Frontal area

Typical values:
- **Upright/hoods**: 0.35-0.40 m²
- **Drops**: 0.30-0.35 m²
- **Aero position**: 0.25-0.30 m²
- **TT position**: 0.20-0.25 m²

In [ ]:
def estimate_cda(height_m, weight_kg, position='drops'):
    """
    Estimate CdA based on rider size and position.
    Uses empirical formulas from cycling literature.
    """
    # Frontal area estimation (Heil, 2001 approximation)
    bsa = 0.00949 * (height_m * 100) ** 0.655 * weight_kg ** 0.441  # Body surface area
    
    # Position multipliers for frontal area
    position_factors = {
        'upright': 0.35,
        'hoods': 0.32,
        'drops': 0.28,
        'aero': 0.24,
        'tt': 0.21
    }
    
    factor = position_factors.get(position, 0.30)
    frontal_area = bsa * factor
    
    # Drag coefficient (fairly constant ~0.7-0.9)
    cd = 0.80
    
    return cd * frontal_area

# Compare positions
HEIGHT = 1.80  # meters
positions = ['upright', 'hoods', 'drops', 'aero', 'tt']

print(f"CdA estimates for {HEIGHT}m, {RIDER_WEIGHT}kg rider:")
for pos in positions:
    cda = estimate_cda(HEIGHT, RIDER_WEIGHT, pos)
    print(f"  {pos:>8}: {cda:.3f} m²")

In [ ]:
def power_for_speed(speed_mps, cda, crr, weight, grade=0):
    """
    Calculate power required for given speed.
    
    Args:
        speed_mps: Speed in m/s
        cda: Drag coefficient × frontal area (m²)
        crr: Rolling resistance coefficient
        weight: Total mass in kg
        grade: Gradient as decimal (0.05 = 5%)
    """
    rho = 1.225  # Air density kg/m³
    g = 9.81     # Gravity m/s²
    
    # Aerodynamic drag
    p_aero = 0.5 * rho * cda * speed_mps ** 3
    
    # Rolling resistance
    p_roll = crr * weight * g * speed_mps
    
    # Gravity (climbing)
    p_gravity = weight * g * grade * speed_mps
    
    return p_aero + p_roll + p_gravity

# Power vs speed at different CdA values
speeds_kph = np.linspace(20, 50, 100)
speeds_mps = speeds_kph / 3.6

fig, ax = plt.subplots(figsize=(12, 6))

CRR = 0.004  # Typical road tire

for pos in ['upright', 'drops', 'aero', 'tt']:
    cda = estimate_cda(HEIGHT, RIDER_WEIGHT, pos)
    powers = [power_for_speed(s, cda, CRR, TOTAL_WEIGHT) for s in speeds_mps]
    ax.plot(speeds_kph, powers, linewidth=2, label=f'{pos} (CdA={cda:.3f})')

ax.axhline(y=FTP, color='orange', linestyle=':', label=f'FTP = {FTP}W')

ax.set_xlabel('Speed (km/h)')
ax.set_ylabel('Power (W)')
ax.set_title('Power Required vs Speed at Different Positions')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(20, 50)
ax.set_ylim(0, 600)

plt.tight_layout()
plt.show()

## 5. Grade Effects on Power

At steep gradients, gravity dominates; aero becomes less important.

In [ ]:
# Power required at constant speed across different gradients
CONSTANT_SPEED = 25 / 3.6  # 25 km/h in m/s
CDA = estimate_cda(HEIGHT, RIDER_WEIGHT, 'drops')

gradients = np.linspace(-10, 15, 100)  # -10% to +15%
powers = [power_for_speed(CONSTANT_SPEED, CDA, CRR, TOTAL_WEIGHT, g/100) for g in gradients]

fig, ax = plt.subplots(figsize=(12, 6))

ax.fill_between(gradients, 0, powers, alpha=0.3)
ax.plot(gradients, powers, 'b-', linewidth=2)
ax.axhline(y=0, color='black', linewidth=0.5)
ax.axhline(y=FTP, color='orange', linestyle=':', label=f'FTP = {FTP}W')
ax.axhline(y=CRITICAL_POWER, color='red', linestyle='--', label=f'CP = {CRITICAL_POWER}W')
ax.axvline(x=0, color='gray', linestyle='-', alpha=0.3)

ax.set_xlabel('Gradient (%)')
ax.set_ylabel('Power (W)')
ax.set_title(f'Power Required at {CONSTANT_SPEED*3.6:.0f} km/h vs Gradient')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Power breakdown at different gradients
test_gradients = [0, 3, 6, 10]

fig, axes = plt.subplots(1, len(test_gradients), figsize=(14, 5))

rho = 1.225
g = 9.81

for ax, grade in zip(axes, test_gradients):
    speed = CONSTANT_SPEED
    
    p_aero = 0.5 * rho * CDA * speed ** 3
    p_roll = CRR * TOTAL_WEIGHT * g * speed
    p_gravity = TOTAL_WEIGHT * g * (grade/100) * speed
    
    components = [p_aero, p_roll, p_gravity]
    labels = ['Aero', 'Rolling', 'Gravity']
    colors = ['skyblue', 'lightgreen', 'salmon']
    
    ax.bar(labels, components, color=colors)
    ax.set_title(f'{grade}% grade\nTotal: {sum(components):.0f}W')
    ax.set_ylabel('Power (W)')
    
    # Show percentages
    total = sum(components)
    for i, (comp, lbl) in enumerate(zip(components, labels)):
        ax.annotate(f'{comp/total*100:.0f}%', (i, comp), 
                   ha='center', va='bottom', fontsize=10)

plt.suptitle(f'Power Breakdown at {CONSTANT_SPEED*3.6:.0f} km/h', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Key Takeaways

- **CP and W'** define your power-duration relationship
- **W' depletion** limits hard efforts; recovery happens below CP
- **CdA** matters most on flat/fast sections
- **Weight** matters most on climbs
- Optimal pacing considers when to spend W' based on terrain